In [ ]:
HF_TOKEN =          # your Hugging Face read token
ASSEMBLYAI_KEY = ""          # optional: free-tier key; leave "" to skip system 2
N_SCAN = 25                  # how many dev clips to scan when choosing
MAX_MIN = 6                  # skip clips longer than this (minutes)

from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
import sys
import os
import subprocess

# Force install dependencies in the current running kernel's environment
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pyannote.audio==3.3.2", "pyannote.metrics==3.2.1",
    "datasets", "soundfile", "requests", "pandas"
])

import io, itertools, soundfile as sf
from datasets import load_dataset, Audio
from pyannote.core import Annotation, Segment

ds = load_dataset("diarizers-community/voxconverse", split="dev", streaming=True)
ds = ds.cast_column("audio", Audio(decode=False))

def build_ref(ex, uri):
    ref = Annotation(uri=uri)
    for s, e, spk in zip(ex["timestamps_start"], ex["timestamps_end"], ex["speakers"]):
        ref[Segment(s, e)] = spk
    return ref

cands = []
for i, ex in enumerate(itertools.islice(ds, N_SCAN)):
    raw = ex["audio"]["bytes"]
    dur = sf.info(io.BytesIO(raw)).duration
    if dur > MAX_MIN * 60:
        continue
    uri = f"clip{i:03d}"
    ref = build_ref(ex, uri)
    ov = ref.get_overlap().duration() / max(ref.get_timeline().support().duration(), 1e-9)
    cands.append(dict(uri=uri, raw=raw, dur=dur, ref=ref,
                      n_spk=len(ref.labels()), overlap=ov))

cands.sort(key=lambda c: (c["n_spk"], c["overlap"]))
picked = [cands[0], cands[len(cands)//2], cands[-1]]   # easy / medium / hard

for c in picked:
    with open(f"{c['uri']}.wav", "wb") as f:
        f.write(c["raw"])
    print(c["uri"], f"{c['dur']:.0f}s", c["n_spk"], "speakers",
          f"overlap {c['overlap']:.1%}")

README.md:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

clip003 28s 1 speakers overlap 0.0%
clip000 179s 4 speakers overlap 3.1%
clip023 181s 10 speakers overlap 0.0%


In [1]:
!pip install "torch==2.8.0" "torchvision==0.23.0" "torchaudio==2.8.0" "huggingface_hub<1.0" "pyannote.audio==3.3.2" "pyannote.metrics==3.2.1" datasets soundfile requests pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of pytorch-metric-learning to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 28.9 MB/s  0:00:21
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 94.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 113.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 49.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 23.6 MB/s  0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 63.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 52.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 25.6 MB/s  0:00:13
   ━━━━━━━━━━

In [3]:
!pip install "transformers<5" "huggingface_hub<1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.0 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]


In [1]:
import torch, torchaudio, huggingface_hub, pyannote.audio, pyannote.metrics
print("torch", torch.__version__, "| torchaudio", torchaudio.__version__)
print("hf_hub", huggingface_hub.__version__, "| pyannote.audio", pyannote.audio.__version__)
print("GPU:", torch.cuda.is_available())

torch 2.8.0+cu128 | torchaudio 2.8.0+cu128
hf_hub 0.36.2 | pyannote.audio 3.3.2
GPU: True


/usr/local/lib/python3.13/dist-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


In [ ]:
HF_TOKEN =           # your Hugging Face read token
ASSEMBLYAI_KEY = ""          # optional: free-tier key; leave "" to skip system 2
N_SCAN = 25                  # how many dev clips to scan when choosing
MAX_MIN = 6                  # skip clips longer than this (minutes)

from huggingface_hub import login
login(HF_TOKEN)

In [11]:
import torch
from torch.torch_version import TorchVersion
from pyannote.audio.core.task import Specifications, Problem

torch.serialization.add_safe_globals([
    TorchVersion,
    Specifications,
    Problem
])

print("PyTorch serialization classes configured!")

PyTorch serialization classes configured!


In [13]:
import os

# Restore legacy checkpoint loading behaviour
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

print("PyTorch checkpoint compatibility enabled!")

PyTorch checkpoint compatibility enabled!


In [14]:
import torch
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=HF_TOKEN
)

if pipeline is None:
    raise RuntimeError("Pipeline loading failed!")

pipeline.to(torch.device("cuda"))

print("Successfully loaded pyannote pipeline!")

/usr/local/lib/python3.13/dist-packages/lightning_fabric/utilities/cloud_io.py:73: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
/usr/local/lib/python3.13/dist-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/221 [00:00<?, ?B/s]

Successfully loaded pyannote pipeline!


In [24]:
import io, itertools, soundfile as sf
from datasets import load_dataset, Audio
from pyannote.core import Annotation, Segment

N_SCAN, MAX_MIN = 50, 6

ds = load_dataset("diarizers-community/voxconverse", split="dev", streaming=True)
ds = ds.cast_column("audio", Audio(decode=False))

def build_ref(ex, uri):
    ref = Annotation(uri=uri)
    for s, e, spk in zip(ex["timestamps_start"], ex["timestamps_end"], ex["speakers"]):
        ref[Segment(s, e)] = spk
    return ref

cands = []
for i, ex in enumerate(itertools.islice(ds, N_SCAN)):
    raw = ex["audio"]["bytes"]
    dur = sf.info(io.BytesIO(raw)).duration
    if dur > MAX_MIN * 60:
        continue
    uri = f"clip{i:03d}"
    ref = build_ref(ex, uri)
    ov = ref.get_overlap().duration() / max(ref.get_timeline().support().duration(), 1e-9)
    cands.append(dict(uri=uri, raw=raw, dur=dur, ref=ref, n_spk=len(ref.labels()), overlap=ov))

cands = [c for c in cands if c["n_spk"] >= 2]
by_ov = sorted(cands, key=lambda c: c["overlap"])
picked = [by_ov[-1], by_ov[len(by_ov)//2], max(cands, key=lambda c: c["n_spk"])]
picked = list({c["uri"]: c for c in picked}.values())

for c in picked:
    open(f"{c['uri']}.wav", "wb").write(c["raw"])
    print(c["uri"], f"{c['dur']:.0f}s", c["n_spk"], "speakers", f"overlap {c['overlap']:.1%}")

clip019 158s 4 speakers overlap 29.0%
clip040 337s 6 speakers overlap 2.2%
clip023 181s 10 speakers overlap 0.0%


In [25]:
print("Pipe:", pipe)
print("Type:", type(pipe))
print("Pipeline variable:", pipeline if 'pipeline' in globals() else "Not defined")

Pipe: <pyannote.audio.pipelines.speaker_diarization.SpeakerDiarization object at 0x7c43cc7d2f10>
Type: <class 'pyannote.audio.pipelines.speaker_diarization.SpeakerDiarization'>
Pipeline variable: <pyannote.audio.pipelines.speaker_diarization.SpeakerDiarization object at 0x7c451bfa5fd0>


In [26]:
import torch
from pyannote.audio import Pipeline

pipe = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=HF_TOKEN
)

if pipe is None:
    raise RuntimeError("Pyannote pipeline initialization failed!")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pipe.to(device)

print("Pipeline initialized successfully!")
print("Device:", device)
print("Pipeline type:", type(pipe))

/usr/local/lib/python3.13/dist-packages/lightning_fabric/utilities/cloud_io.py:73: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
/usr/local/lib/python3.13/dist-packages/pyannote/audio/core/io.py:212: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  torchaudio.list_audio_backends()


Pipeline initialized successfully!
Device: cuda
Pipeline type: <class 'pyannote.audio.pipelines.speaker_diarization.SpeakerDiarization'>


In [27]:
import torch
import pyannote.audio

print("PyTorch:", torch.__version__)
print("Pyannote:", pyannote.audio.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.8.0+cu128
Pyannote: 3.3.2
CUDA available: True


In [28]:
hyps = {}

for c in picked:
    wav = f"{c['uri']}.wav"

    # Pyannote 3.1 (blind)
    hyps[("pyannote-3.1 (blind)", c["uri"])] = pipe(wav)

    # Pyannote 3.1 (oracle speaker count)
    hyps[("pyannote-3.1 (oracle #spk)", c["uri"])] = pipe(
        wav,
        num_speakers=c["n_spk"]
    )

print("Done:", len(hyps), "runs")

/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)
/usr/local/lib/python3.13/dist-packages/pyannote/audio/core/io.py:85: UserWarning: torchaudio._backend.utils.info has been

Done: 6 runs


In [29]:
import pandas as pd
from pyannote.core import Timeline
from pyannote.metrics.diarization import DiarizationErrorRate, JaccardErrorRate

rows = []
for (system, uri), hyp in hyps.items():
    c = next(x for x in picked if x["uri"] == uri)
    uem = Timeline([Segment(0, c["dur"])])
    row = dict(clip=uri, system=system, dur_s=round(c["dur"]), ref_spk=c["n_spk"],
               hyp_spk=len(hyp.labels()), overlap=f"{c['overlap']:.1%}")
    for name, kw in {"strict": dict(collar=0.0, skip_overlap=False),
                     "lenient": dict(collar=0.25, skip_overlap=True)}.items():
        d = DiarizationErrorRate(**kw)(c["ref"], hyp, uem=uem, detailed=True)
        row[f"DER_{name}"] = round(100 * d["diarization error rate"], 1)
        if name == "strict":
            row["miss%"] = round(100 * d["missed detection"] / d["total"], 1)
            row["FA%"] = round(100 * d["false alarm"] / d["total"], 1)
            row["conf%"] = round(100 * d["confusion"] / d["total"], 1)
    row["JER_strict"] = round(100 * JaccardErrorRate(collar=0.0, skip_overlap=False)(c["ref"], hyp, uem=uem), 1)
    rows.append(row)

res = pd.DataFrame(rows).sort_values(["clip", "system"])
display(res); res.to_csv("results.csv", index=False)

,clip,system,dur_s,ref_spk,hyp_spk,overlap,DER_strict,miss%,FA%,conf%,DER_lenient,JER_strict
0,clip019,pyannote-3.1 (blind),158,4,3,29.0%,20.2,3.9,3.9,12.3,6.3,53.9
1,clip019,pyannote-3.1 (oracle #spk),158,4,4,29.0%,27.7,3.9,3.9,19.9,8.7,53.4
4,clip023,pyannote-3.1 (blind),181,10,10,0.0%,5.9,3.3,1.0,1.6,3.8,7.3
5,clip023,pyannote-3.1 (oracle #spk),181,10,10,0.0%,5.9,3.3,1.0,1.6,3.8,7.3
2,clip040,pyannote-3.1 (blind),337,6,6,2.2%,3.6,2.0,0.5,1.1,1.3,7.6
3,clip040,pyannote-3.1 (oracle #spk),337,6,6,2.2%,3.6,2.0,0.5,1.1,1.3,7.6


In [30]:
import platform, datetime
log = f"""date: {datetime.date.today()}
python: {platform.python_version()}
torch: {torch.__version__} | torchaudio: {torchaudio.__version__}
pyannote.audio: {pyannote.audio.__version__} | pyannote.metrics: 3.2.1
pipeline: pyannote/speaker-diarization-3.1
dataset: diarizers-community/voxconverse (dev), clips: {[c['uri'] for c in picked]}
strict: collar=0.0, skip_overlap=False | lenient: collar=0.25, skip_overlap=True
"""
open("run_log.txt", "w").write(log); print(log)

from google.colab import files
files.download("results.csv"); files.download("run_log.txt")

date: 2026-11-11
python: 3.13.16
torch: 2.8.0+cu128 | torchaudio: 2.8.0+cu128
pyannote.audio: 3.3.2 | pyannote.metrics: 3.2.1
pipeline: pyannote/speaker-diarization-3.1
dataset: diarizers-community/voxconverse (dev), clips: ['clip019', 'clip040', 'clip023']
strict: collar=0.0, skip_overlap=False | lenient: collar=0.25, skip_overlap=True



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>